In [16]:
import datetime as dt
import os
import time
 
import matplotlib
 
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

CONFIGURAÇÃO

Tudo isso será importante para a consulta da API (Claude que fez)

In [17]:
NIVEL = "n6"                # n6 = municipio | n9 = microrregiao | n3 = UF
CODIGO_REGIAO = "3148004"   # Patos de Minas (codigo IBGE do municipio)
LAT, LON = -18.5789, -46.5183   # coordenadas aproximadas de Patos de Minas
 
# Tabelas tentadas em ordem (a primeira que devolver dados e usada)
TABELAS = [5457, 1612]
CULTURAS = {"Milho": "Milho (em grão)", "Soja": "Soja (em grão)"}
VAR_RENDIMENTO = "112"      # rendimento medio da producao (kg/ha)
 
# Ano agricola: a safra colhida no ano Y usa a chuva de out(Y-1) a abr(Y)
MESES_SAFRA = [10, 11, 12, 1, 2, 3, 4]
NOMES_MES = {10: "Out(-1)", 11: "Nov(-1)", 12: "Dez(-1)", 1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr"}
 
# Janelas fenologicas aproximadas (ajuste conforme o ciclo/epoca de plantio da regiao)
JANELAS = {
    "Soja: Jan-Mar (enchimento)": [1, 2, 3],
    "Milho: Dez-Fev (floracao/enchimento)": [12, 1, 2],
}
 
ANO_MIN_CHUVA = 1981        # a NASA POWER (MERRA-2) comeca em 1981
PASTA_SAIDA = "saida"
 
SIDRA = "https://apisidra.ibge.gov.br/values"
META = "https://servicodados.ibge.gov.br/api/v3/agregados/{t}/metadados"
POWER = "https://power.larc.nasa.gov/api/temporal/daily/point"

Funções

In [18]:
def get_json(url, params=None, tentativas=3):
    for i in range(tentativas):
        try:
            r = requests.get(url, params=params, timeout=90)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if i == tentativas - 1:
                raise
            time.sleep(2 * (i + 1))

In [19]:
def achar_classificacao(tabela):
    """Descobre, nos metadados da tabela, o codigo da classificacao de produtos
    e os codigos de milho e soja (evita depender de codigos decorados)."""
    meta = get_json(META.format(t=tabela))
    for cls in meta.get("classificacoes", []):
        cats = {c["nome"].strip(): c["id"] for c in cls.get("categorias", [])}
        achados = {k: cats[v] for k, v in CULTURAS.items() if v in cats}
        if len(achados) == len(CULTURAS):
            return str(cls["id"]), achados
    raise ValueError(f"Tabela {tabela}: nao encontrei milho/soja nas classificacoes.")

In [20]:
def baixar_produtividade(tabela):
    cls, cats = achar_classificacao(tabela)
    ids = ",".join(str(i) for i in cats.values())
    url = f"{SIDRA}/t/{tabela}/{NIVEL}/{CODIGO_REGIAO}/v/{VAR_RENDIMENTO}/p/all/c{cls}/{ids}"
    dados = get_json(url)
    cab, linhas = dados[0], dados[1:]
    if not linhas:
        raise ValueError(f"Tabela {tabela}: resposta vazia.")
 
    # A 1a linha descreve as colunas; localizamos ano e produto pelo nome
    k_ano = next(k for k, v in cab.items() if v == "Ano")
    k_prod = next(k for k, v in cab.items() if v.startswith("Produto") and not v.endswith("(Código)"))
    inverso = {v: k for k, v in CULTURAS.items()}
 
    df = pd.DataFrame(
        {
            "ano": [int(l[k_ano]) for l in linhas],
            "cultura": [inverso.get(l[k_prod].strip(), l[k_prod]) for l in linhas],
            # "-", "..", "..." e "X" (sem dado / sigilo) viram NaN
            "rend_kg_ha": pd.to_numeric(pd.Series([l["V"] for l in linhas]), errors="coerce"),
        }
    )
    rend = df.pivot_table(index="ano", columns="cultura", values="rend_kg_ha")
    rend = rend.dropna(how="all")
    if rend.empty:
        raise ValueError(f"Tabela {tabela}: sem valores numericos para a regiao.")
    return rend

In [21]:
def produtividade():
    erros = []
    for t in TABELAS:
        try:
            rend = baixar_produtividade(t)
            print(f"[SIDRA] tabela {t}: {rend.index.min()}-{rend.index.max()}")
            return rend
        except Exception as e:  # tenta a proxima tabela
            erros.append(f"tabela {t}: {e}")
    raise RuntimeError("Nao consegui baixar a produtividade.\n" + "\n".join(erros))

In [22]:
def baixar_chuva(ano_ini, ano_fim):
    limite = dt.date.today() - dt.timedelta(days=10)  # a NASA tem alguns dias de atraso
    partes = []
    for a in range(ano_ini, ano_fim + 1, 5):  # blocos de 5 anos
        b = min(a + 4, ano_fim)
        fim = min(dt.date(b, 12, 31), limite)
        if dt.date(a, 1, 1) > fim:
            break
        j = get_json(
            POWER,
            params={
                "parameters": "PRECTOTCORR",
                "community": "AG",
                "longitude": LON,
                "latitude": LAT,
                "start": f"{a}0101",
                "end": fim.strftime("%Y%m%d"),
                "format": "JSON",
            },
        )
        partes.append(pd.Series(j["properties"]["parameter"]["PRECTOTCORR"]))
    s = pd.concat(partes)
    s.index = pd.to_datetime(s.index, format="%Y%m%d")
    s = s.astype(float).where(lambda x: x > -990)  # -999 = dado ausente
    print(f"[NASA POWER] {s.index.min().date()} a {s.index.max().date()} ({len(s)} dias)")
    return s

In [23]:
def chuva_mensal(diaria):
    # min_count evita somar meses com muitos dias faltando
    return diaria.resample("MS").sum(min_count=25)

In [24]:
def montar_features(mensal, anos_colheita):
    """Uma linha por ano de colheita Y, com a chuva (mm) de cada mes do ano agricola."""
    linhas = {}
    for y in anos_colheita:
        reg = {}
        for m in MESES_SAFRA:
            ref = pd.Timestamp(year=y - 1 if m >= 10 else y, month=m, day=1)
            reg[NOMES_MES[m]] = mensal.get(ref, np.nan)
        linhas[y] = reg
    X = pd.DataFrame.from_dict(linhas, orient="index")
    X["Safra Out-Mar"] = X[[NOMES_MES[m] for m in [10, 11, 12, 1, 2, 3]]].sum(axis=1, min_count=6)
    for nome, meses in JANELAS.items():
        X[nome] = X[[NOMES_MES[m] for m in meses]].sum(axis=1, min_count=len(meses))
    return X

In [25]:
def correlacionar(rend, X):
    """Pearson/Spearman entre chuva e produtividade, bruta e sem tendencia.
    A produtividade sobe com a tecnologia ao longo dos anos; tirar a tendencia
    linear isola o efeito do clima de cada ano."""
    resultados = []
    for cultura in rend.columns:
        y_all = rend[cultura].dropna()
        anos = y_all.index.intersection(X.index)
        y = y_all.loc[anos]
        if len(y) < 8:
            continue
        t = anos.values - anos.values.mean()
        y_det = y.values - np.polyval(np.polyfit(t, y.values, 1), t)
        for feat in X.columns:
            x = X.loc[anos, feat].values
            ok = ~np.isnan(x)
            if ok.sum() < 8:
                continue
            r, p = stats.pearsonr(x[ok], y.values[ok])
            rd, pd_ = stats.pearsonr(x[ok], y_det[ok])
            rho, ps = stats.spearmanr(x[ok], y_det[ok])
            resultados.append(
                {
                    "cultura": cultura,
                    "periodo_chuva": feat,
                    "n_anos": int(ok.sum()),
                    "pearson_bruta": r,
                    "p_bruta": p,
                    "pearson_sem_tendencia": rd,
                    "p_sem_tendencia": pd_,
                    "spearman_sem_tendencia": rho,
                    "p_spearman": ps,
                }
            )
    return pd.DataFrame(resultados)

In [26]:
def grafico(rend, X, corr, caminho):
    """Histogramas: um do rendimento de cada cultura e um da chuva da safra (out-mar).
    (corr continua na assinatura so para nao mudar a chamada em main().)"""
    culturas = list(rend.columns)
    fig, eixos = plt.subplots(1, len(culturas) + 1, figsize=(5.5 * (len(culturas) + 1), 4.5), squeeze=False)
    eixos = eixos[0]
 
    # Um histograma de rendimento por cultura
    for ax, c in zip(eixos, culturas):
        dados = rend[c].dropna()
        ax.hist(dados, bins="auto", color="#2a7f62", edgecolor="white")
        ax.axvline(dados.mean(), color="k", ls="--", lw=1, label=f"media: {dados.mean():.0f}")
        ax.set_title(f"{c}: distribuicao do rendimento ({len(dados)} anos)")
        ax.set_xlabel("Rendimento (kg/ha)")
        ax.set_ylabel("Numero de anos")
        ax.legend()
 
    # Histograma da chuva acumulada da safra
    chuva = X["Safra Out-Mar"].dropna()
    ax = eixos[-1]
    ax.hist(chuva, bins="auto", color="#3b6ea5", edgecolor="white")
    ax.axvline(chuva.mean(), color="k", ls="--", lw=1, label=f"media: {chuva.mean():.0f}")
    ax.set_title(f"Chuva da safra out-mar ({len(chuva)} anos)")
    ax.set_xlabel("Chuva acumulada (mm)")
    ax.set_ylabel("Numero de anos")
    ax.legend()
 
    fig.tight_layout()
    fig.savefig(caminho, dpi=130)
    plt.close(fig)

In [27]:
def main():
    os.makedirs(PASTA_SAIDA, exist_ok=True)
 
    rend = produtividade()
    ano_ini = max(ANO_MIN_CHUVA, int(rend.index.min()) - 1)
    diaria = baixar_chuva(ano_ini, int(rend.index.max()))
    mensal = chuva_mensal(diaria)
 
    X = montar_features(mensal, rend.index)
    corr = correlacionar(rend, X)
 
    rend.to_csv(f"{PASTA_SAIDA}/produtividade.csv")
    X.round(1).to_csv(f"{PASTA_SAIDA}/chuva_mensal.csv")
    corr.round(3).to_csv(f"{PASTA_SAIDA}/correlacoes.csv", index=False)
    grafico(rend, X, corr, f"{PASTA_SAIDA}/grafico.png")
 
    pd.set_option("display.width", 160)
    print("\nCorrelacao chuva x rendimento (r sem tendencia; p < 0,05 = significativo):")
    cols = ["cultura", "periodo_chuva", "n_anos", "pearson_bruta", "pearson_sem_tendencia", "p_sem_tendencia"]
    print(corr[cols].round(3).to_string(index=False))
    print(f"\nArquivos gravados em ./{PASTA_SAIDA}/")

In [28]:
main()

[SIDRA] tabela 5457: 1974-2025
[NASA POWER] 1981-01-01 a 2025-12-31 (16436 dias)

Correlacao chuva x rendimento (r sem tendencia; p < 0,05 = significativo):
cultura                        periodo_chuva  n_anos  pearson_bruta  pearson_sem_tendencia  p_sem_tendencia
  Milho                              Out(-1)      44         -0.309                 -0.043            0.779
  Milho                              Nov(-1)      44         -0.283                 -0.201            0.190
  Milho                              Dez(-1)      44         -0.493                 -0.195            0.205
  Milho                                  Jan      45         -0.317                  0.084            0.581
  Milho                                  Fev      45         -0.242                 -0.379            0.010
  Milho                                  Mar      45         -0.340                  0.071            0.642
  Milho                                  Abr      45         -0.130                  0.